<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/iLogos/logo_novafct.png" style="width: 30%;"/>

## Mecânica dos Sólidos II

## Métodos energéticos

### Problema 4

As barras da treliça indicada são feitas de aço e têm a área da secção transversal representada. Considerando $E = 200$ GPa, determine os deslocamentos horizontal e vertical do nó C quando a força de 30 kN é aplicada.

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/Notebooks/Au10/P4/MSII_Au10_P4.png"
style="width: 40%;"/>

In [13]:
import numpy as np
from sympy import symbols, Eq
from sympy.solvers import solve
import matplotlib.pyplot as plt
from dataclasses import dataclass

cor = '1'
if cor == '1':
    plt.rcParams['axes.facecolor'] = (.15, .15, .15)
    plt.rcParams['figure.facecolor'] = (.15, .15, .15)
    plt.rcParams['font.family'] = 'monospace'
    plt.rcParams['font.size'] = 18
    # plt.rcParams['text.usetex'] = True
    params = {"ytick.color" : (.8, .8, .8),
              "xtick.color" : (.8, .8, .8),
              "grid.color" : (.2, .2, .2),
              "text.color" : (.7, .7, .7),
              "axes.labelcolor" : (.8, .8, .8),
              "axes.edgecolor" : (.15, .15, .15)}
else:
    plt.rcParams['axes.facecolor'] = (.7, .7, .7)
    plt.rcParams['figure.facecolor'] = (.7, .7, .7)
    plt.rcParams['font.family'] = 'monospace'
    plt.rcParams['font.size'] = 18
    # plt.rcParams['text.usetex'] = True
    params = {"ytick.color" : (.1, .1, .1),
              "xtick.color" : (.1, .1, .1),
              "grid.color" : (.2, .2, .2),
              "text.color" : (.1, .1, .1),
              "axes.labelcolor" : (.1, .1, .1),
              "axes.edgecolor" : (.15, .15, .15)}
plt.rcParams.update(params)

# data structure, units: N, mm, MPa
from dataclasses import dataclass

@dataclass
class Varin:
    LAB: float  # unit: m
    LBCh: float  # unit: m
    LBCv: float  # unit: m
    E: float  # unit: Pa
    AAB: float  # unit: m^2
    ABC: float  # unit: m^2
    AAC: float  # unit: m^2
    P: float  # unit: N

# Initialize the data structure with the given values
d = Varin(LAB=1.2, LBCh=1.5, LBCv=1.2, E=200e9, AAB=2500e-6, ABC=3750e-6, AAC=1250e-6, P=30e3)
LAC = np.sqrt((d.LAB + d.LBCh)**2 + d.LBCv**2)
print(f'{LAC = :.2f} [m]')
LBC = np.sqrt(d.LBCh**2 + d.LBCv**2)
print(f'{LBC = :.2f} [m]')

angBAC = np.arctan(d.LBCv/(d.LAB + d.LBCh))
print(f'{angBAC = :.2f} [rad] = {np.rad2deg(angBAC):.2f} [deg]')
anghBC = np.arctan(d.LBCv/d.LBCh)
print(f'{anghBC = :.2f} [rad] = {np.rad2deg(anghBC):.2f} [deg]')


LAC = 2.95 [m]
LBC = 1.92 [m]
angBAC = 0.42 [rad] = 23.96 [deg]
anghBC = 0.67 [rad] = 38.66 [deg]


## Resolução

- Cálculo do deslocamento do ponto de aplicação da força $P$:

\begin{equation*}
\delta_P = \frac{\partial U}{\partial P} = \frac{\partial}{\partial P}\left(\sum_i \frac{F_i²L_i}{2A_iE}\right)
=  \frac{1}{E}\sum_i \frac{F_i L_i}{A_i} \frac{\partial F_i}{\partial P}
\end{equation*}


- Cálculo dos esforços das barras pelo método dos nós:

In [14]:
fac, fbc, p, fab = symbols('fac fbc p fab')

# Nó C
print('Nó C')

C_sumFX = Eq(-fac*np.cos(angBAC) - fbc*np.cos(anghBC), 0)
print(f'{C_sumFX = }')
C_sumFY = Eq(-fac*np.sin(angBAC) - fbc*np.sin(anghBC) - p, 0)
print(f'{C_sumFY = }')

# Nó B
print('\nNó B')
B_sumFX = Eq(-fab + fbc*np.cos(anghBC), 0)
print(f'{B_sumFX = }\n')

def calculo_esforcos(load):

    sol = solve([C_sumFX,C_sumFY, B_sumFX],[fac, fbc, fab])
    fac_p = sol[fac]
    fbc_p = sol[fbc]
    fab_p = sol[fab]

    fac_p_eval = fac_p.subs(p,load)
    fbc_p_eval = fbc_p.subs(p,load)
    fab_p_eval = fab_p.subs(p,load)

    print(f'{fac_p = } [N] | fac_p_eval = {fac_p_eval*1e-3:.2f} [kN]')
    print(f'{fbc_p = } [N] | fbc_p_eval = {fbc_p_eval*1e-3:.2f} [kN]')
    print(f'{fab_p = } [N] | fab_p_eval = {fab_p_eval*1e-3:.2f} [kN]')

    return fac_p_eval, fbc_p_eval, fab_p_eval

FAC, FBC, FAB = calculo_esforcos(d.P)

Nó C
C_sumFX = Eq(-0.913811548620257*fac - 0.78086880944303*fbc, 0)
C_sumFY = Eq(-0.406138466053448*fac - 0.624695047554424*fbc - p, 0)

Nó B
B_sumFX = Eq(-fab + 0.78086880944303*fbc, 0)

fac_p = 3.07776806306129*p [N] | fac_p_eval = 92.33 [kN]
fbc_p = -3.60175738355598*p [N] | fbc_p_eval = -108.05 [kN]
fab_p = -2.8125*p [N] | fab_p_eval = -84.38 [kN]


In [15]:
FACdP, FBCdP, FABdP = calculo_esforcos(1)

fac_p = 3.07776806306129*p [N] | fac_p_eval = 0.00 [kN]
fbc_p = -3.60175738355598*p [N] | fbc_p_eval = -0.00 [kN]
fab_p = -2.8125*p [N] | fab_p_eval = -0.00 [kN]


In [16]:
deslv = ((LAC/d.AAC)*(FAC*FACdP) + (LBC/d.ABC)*(FBC*FBCdP) + (d.LAB/d.AAB)*(FAB*FABdP))/d.E
print(f'{deslv = } [m] = {deslv*1e3:.3f} [mm]')

deslv = 0.00492493200471476 [m] = 4.925 [mm]


- Cálculo do deslocamento horizontal em C:

Considere-se agora uma força fictícia $Q$ no ponto e na direção onde se pretende avaliar o deslocamento, i.e., força horizontal no ponto C.

\begin{equation*}
\delta_Q  =  \frac{1}{E}\sum_i \frac{L_i}{A_i} F_i F_{ui}
\end{equation*}

In [17]:
print('Carregamento unitário :: Q = 1 N no ponto e direção do deslocamento horizontal')
fac_, fbc_, fab_, q = symbols('fac_ fbc_ fab_ q')

# Nó C
print('Nó C')

C_sumFX_ = Eq(-fac_*np.cos(angBAC) - fbc_*np.cos(anghBC) + q, 0)
print(f'{C_sumFX_ = }')
C_sumFY_ = Eq(-fac_*np.sin(angBAC) - fbc_*np.sin(anghBC), 0)
print(f'{C_sumFY_ = }')

# Nó B
print('\nNó B')
B_sumFX_ = Eq(-fab_ + fbc_*np.cos(anghBC), 0)
print(f'{B_sumFX_ = }\n')

def calculo_esforcos_U(load):

    sol = solve([C_sumFX_, C_sumFY_, B_sumFX_],[fac_, fbc_, fab_])
    fac_q = sol[fac_]
    fbc_q = sol[fbc_]
    fab_q = sol[fab_]

    fac_q_eval = fac_q.subs(q,load)
    fbc_q_eval = fbc_q.subs(q,load)
    fab_q_eval = fab_q.subs(q,load)

    print(f'\n{fac_q = } [N] | fac_p_eval = {fac_q_eval*1e-3:.2e} [kN]')
    print(f'{fbc_q = } [N] | fbc_p_eval = {fbc_q_eval*1e-3:.2e} [kN]')
    print(f'{fab_q = } [N] | fab_p_eval = {fab_q_eval*1e-3:.2e} [kN]')

    return fac_q_eval, fbc_q_eval, fab_q_eval

FAC_, FBC_, FAB_ = calculo_esforcos(load=d.P)
FAC_U, FBC_U, FAB_U = calculo_esforcos_U(load=1)
deslh = ((LAC/d.AAC)*(FAC_*FAC_U) + (LBC/d.ABC)*(FBC_*FBC_U) + (d.LAB/d.AAB)*(FAB_*FAB_U))/d.E
print(f'{deslh = } [m] = {deslh*1e3:.3f} [mm]')

Carregamento unitário :: Q = 1 N no ponto e direção do deslocamento horizontal
Nó C
C_sumFX_ = Eq(-0.913811548620257*fac_ - 0.78086880944303*fbc_ + q, 0)
C_sumFY_ = Eq(-0.406138466053448*fac_ - 0.624695047554424*fbc_, 0)

Nó B
B_sumFX_ = Eq(-fab_ + 0.78086880944303*fbc_, 0)

fac_p = 3.07776806306129*p [N] | fac_p_eval = 92.33 [kN]
fbc_p = -3.60175738355598*p [N] | fbc_p_eval = -108.05 [kN]
fab_p = -2.8125*p [N] | fab_p_eval = -84.38 [kN]

fac_q = 2.46221445044903*q [N] | fac_p_eval = 2.46e-3 [kN]
fbc_q = -1.60078105935822*q [N] | fbc_p_eval = -1.60e-3 [kN]
fab_q = -1.25*q [N] | fab_p_eval = -1.25e-3 [kN]
deslh = 0.00338303267722990 [m] = 3.383 [mm]


# Apêndice: Ftool solution


<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/Notebooks/Au10/P4/N.png"
style="width: 60%;"/>

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/Notebooks/Au10/P4/Def.png"
style="width: 60%;"/>


---

Copyright (c) DEMI - FCT NOVA

Interactive computing by <a href="https://jupyter.org/" target="_blank"> <span
style="color:#333399"> Jupyter Notebook </span> </a> &nbsp;|&nbsp;Coded by <a href = "mailto: jmc.xavier@fct.unl.pt">José Xavier</a>

Licensed under  <a href="http://creativecommons.org/licenses/by-sa/4.0/"
target="_blank"> <span style="color:#333399;font-size: 20px"> CC BY-SA 4.0  </span></a>